In [1]:
#Install and load necessary packages
install.packages(c("foreach", "doParallel", "MASS"))
library(foreach)
library(doParallel)
#library(tictoc) # for timing the simulation

# 1. Define the core simulation function



# --- Helpers ---------------------------------------------------------------
beta_params_from_mean_cv <- function(m, CV){
  a <- (1 - m)/CV^2 - m
  if (a <= 0) stop("Infeasible (alpha<=0). Decrease CV or increase mean (>0.5 for left skew).")
  b <- a * (1 - m) / m
  if (a <= b) warning("alpha <= beta (may not be left-skewed). Ensure mean > 0.5.")
  list(alpha = a, beta = b)
}

calib_latent_rho <- function(r_target, qX, qY, N = 1e6, seed = 1) {
  if (r_target == 0) return(0)
  set.seed(seed)
  N1 <- rnorm(N); N2 <- rnorm(N)
  X <- qX(pnorm(N1))
  f <- function(rho) cor(X, qY(pnorm(rho * N1 + sqrt(1 - rho^2) * N2))) - r_target
  uniroot(f, c(-0.999, 0.999), tol = 1e-6)$root
}

# (Gaussian copula)
gen_copula <- function(n, rho_latent, qX, qY, seed) {
  set.seed(seed)
  N1 <- rnorm(n); N2 <- rnorm(n)
  list(X = qX(pnorm(N1)),
       Y = qY(pnorm(rho_latent * N1 + sqrt(1 - rho_latent^2) * N2)))
}

p1 <- beta_params_from_mean_cv(0.70, 0.25)    # Beta(4.1000, 1.7571)
p2 <- beta_params_from_mean_cv(0.85, 0.25)    # Beta(1.5500, 0.2735)
qX <- function(u) qbeta(u, p1$alpha, p1$beta)
qY <- function(u) qbeta(u, p2$alpha, p2$beta)
true_delta <- 0



# Profiled negative log-likelihood under H0 (shared m with delta shift)
nll0_profile <- function(par, x, y, delta) {
  lsX <- par[1]; lsY <- par[2]; zr <- par[3]
  sX  <- exp(lsX)
  sY  <- exp(lsY)
  rho <- tanh(zr)

  # profiled m-hat given (sX,sY) and delta
  mhat <- 0.5 * (mean(x)/sX + mean(y)/sY - delta)

  # standardized residuals
  rx <- x/sX - mhat
  ry <- y/sY - (mhat + delta)

  one_m_r2 <- 1 - rho^2
  n <- length(x)

  quad <- sum(rx^2 - 2*rho*rx*ry + ry^2) / one_m_r2
  logdet <- 2*log(sX) + 2*log(sY) + log(one_m_r2)

  0.5 * (n*log(2*pi) + n*logdet + quad)
}

# Profiled negative log-likelihood under H1 (separate means)
nll1_profile <- function(par, x, y) {
  lsX <- par[1]; lsY <- par[2]; zr <- par[3]
  sX  <- exp(lsX)
  sY  <- exp(lsY)
  rho <- tanh(zr)

  # profiled mX-hat, mY-hat
  mXhat <- mean(x)/sX
  mYhat <- mean(y)/sY

  rx <- x/sX - mXhat
  ry <- y/sY - mYhat

  one_m_r2 <- 1 - rho^2
  n <- length(x)

  quad <- sum(rx^2 - 2*rho*rx*ry + ry^2) / one_m_r2
  logdet <- 2*log(sX) + 2*log(sY) + log(one_m_r2)

  0.5 * (n*log(2*pi) + n*logdet + quad)
}





#Part 2: Simmulation ###################################################################################

# 2. Set up the parallel backend
# Detect the number of available cores and use them (leaving one free for system tasks)
n_cores <- detectCores()
if (n_cores > 1) {
  # Create a cluster
  cl <- makeCluster(n_cores - 1)
  registerDoParallel(cl)
  cat(paste("Registered parallel backend with", n_cores - 1, "cores.\n"))
} else {
  registerDoSEQ() # Fallback to sequential if only one core is available
  cat("Only one core detected, running sequentially.\n")
}

# 3. Run the long simulation in parallel

nsim <- 1000 # A large number for a long simulation
cat(paste("Starting", nsim, "simulations...\n"))

n_sample_size=c(30,100,500,1000) #Sample Size
rho_correlation=c(-.7,-.3,0,.3,.7) #Correlation
#nsim=2 #Number of Simulation

Out_Matrix<-matrix(NA, nrow=20, ncol=6)


t0<-proc.time() # Start timing

for(i1 in 1:length(rho_correlation)){
  rho_lat <- calib_latent_rho(rho_correlation[i1], qX, qY)
    for(i2 in 1:length(n_sample_size)){


# The %dopar% operator tells foreach to run the iterations in parallel
results <- foreach(kk = 1:nsim, .combine = rbind) %dopar% {

n<-n_sample_size[i2]

set <- gen_copula(n, rho_lat, qX, qY, seed = 2025 + kk)
xt <- set$X
yt <- set$Y

xtBar<-mean(xt)
ytBar<-mean(yt)

# good starting values from sample moments
    init_sX  <- sd(xt)
    init_sY  <- sd(yt)
    init_rho <- cor(xt, yt)
    par0 <- c(log(init_sX), log(init_sY), atanh(pmax(pmin(init_rho, 0.99), -0.99)))

    crit <- qchisq(.95, 1)
    true_delta <- 0   # Model 1: eta_Y - eta_X = 0

    # H1:
    op1 <- optim(par0, nll1_profile, x = xt, y = yt, method = "Nelder-Mead",
                 control = list(maxit = 4000, reltol = 1e-10))

    # LRT(delta) - crit; if it fails: NA
    g <- function(d) {
      op0 <- optim(par0, nll0_profile, x = xt, y = yt, delta = d, method = "Nelder-Mead",
                   control = list(maxit = 4000, reltol = 1e-10))
      if (op0$convergence != 0) return(NA)
      2 * (op0$value - op1$value) - crit
    }

    # MLE of delta (= eta_Y - eta_X)
    sXm <- sqrt(mean((xt - mean(xt))^2)); sYm <- sqrt(mean((yt - mean(yt))^2))
    d_hat <- mean(yt)/sYm - mean(xt)/sXm

    find_end <- function(dir, max_step = 2^10) {
      step <- 1
      repeat {
        b  <- d_hat + dir * step
        gb <- g(b)
        if (is.na(gb)) return(NA)
        if (gb > 0) break
        step <- 2 * step
        if (step > max_step) return(NA)
      }
      uniroot(g, sort(c(d_hat, b)), tol = 1e-6)$root
    }

    Lower_CI <- find_end(-1)
    Upper_CI <- find_end(+1)

    if (op1$convergence != 0 || is.na(Lower_CI) || is.na(Upper_CI)) {
      data.frame(sim_id = kk, Cov.out = NA, Len_out = NA,
                 Lower = NA, Upper = NA, fail = 1)
    } else {
      data.frame(sim_id = kk,
                 Cov.out = as.numeric(Lower_CI <= true_delta & true_delta <= Upper_CI),
                 Len_out = Upper_CI - Lower_CI,
                 Lower = Lower_CI, Upper = Upper_CI, fail = 0)
    }
} # End of foreach

        Cov_rate_out<-results[,2]
        Length_CI_out<-results[,3]
        Cov_rate_out[is.infinite(Cov_rate_out)]<-NA
        Length_CI_out[is.infinite(Length_CI_out)]<-NA
        Cov_rate_out<-na.omit(Cov_rate_out)
        Length_CI_out<-na.omit(Length_CI_out)


        ok <- results$fail == 0                      # only successes
        cov_i  <- results$Cov.out[ok]
        len_i  <- results$Len_out[ok]
        low_i  <- results$Lower[ok]
        up_i   <- results$Upper[ok]
        R      <- sum(ok)                            # number of "s"

        OM <- (i1 - 1) * length(n_sample_size) + i2
       Out_Matrix[OM, 1] <- mean(cov_i)                              # coverage
       Out_Matrix[OM, 2] <- round(mean(len_i), 4)                    # ave. length
       Out_Matrix[OM, 3] <- sum(results$fail)                        # number of "f"
       Out_Matrix[OM, 4] <- round(sqrt(mean(cov_i) * (1 - mean(cov_i)) / R), 4)  # coverage의 MC SE
       Out_Matrix[OM, 5] <- mean(true_delta < low_i)                 # Lower miss
       Out_Matrix[OM, 6] <- mean(true_delta > up_i)                  # Upper miss

    }
}



# 4. Stop the cluster and analyze results
if (exists("cl")) {
  stopCluster(cl) # Important: always stop the cluster when done
}


t1<-proc.time() # Stop timing

t1-t0
Out_Matrix




Installing packages into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependency ‘iterators’


Loading required package: iterators

Loading required package: parallel



Registered parallel backend with 7 cores.
Starting 1000 simulations...


   user  system elapsed 
 53.581   1.440 374.728 

0.704,2.6200,0,0.0144,0.173,0.123
0.769,1.3644,0,0.0133,0.143,0.088
0.745,0.5985,0,0.0138,0.130,0.125
0.762,0.4222,0,0.0135,0.120,0.118
0.779,3.1614,0,0.0131,0.136,0.085
0.812,1.6544,0,0.0124,0.108,0.080
0.789,0.7294,0,0.0129,0.115,0.096
0.821,0.5140,0,0.0121,0.074,0.105
0.810,3.2269,0,0.0124,0.123,0.067
0.832,1.6861,0,0.0118,0.093,0.075
0.811,0.7467,0,0.0124,0.110,0.079
